In [ ]:
# ============================================================
# HIPE-2026 OCR Cleaning Pipeline
# Applies LLM-based correction to all dataset splits
# ============================================================

# 1. SETUP
!pip install openai groq

from openai import OpenAI
from google.colab import userdata, drive
import json, re, copy, time

drive.mount('/content/drive')

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_KEY')
)

In [ ]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def call_with_retry(client, model, messages, max_tokens):
    """API call with automatic retry on rate limit errors."""
    while True:
        try:
            return client.chat.completions.create(
                model=model, messages=messages, max_tokens=max_tokens
            )
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                print("Rate limit hit, waiting 15 seconds...")
                time.sleep(15)
            else:
                raise e

def clean_text_llm(text, client, language="en"):
    """
    Cleans OCR artifacts from historical newspaper text using Llama 3.3 70B.
    Applies structural fixes first, then LLM-based correction chunk by chunk.
    """
    # Structural fixes
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)  # fix hyphenated line breaks
    text = re.sub(r'\n', ' ', text)                # newlines to spaces
    text = re.sub(r' +', ' ', text)                # collapse multiple spaces

    # Chunk-based LLM correction
    chunk_size = 1000
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    corrected = []

    for i, chunk in enumerate(chunks):
        if len(chunk.strip()) < 10:
            corrected.append(chunk)
            continue

        print(f"  chunk {i+1}/{len(chunks)}", end="\r")

        response = call_with_retry(client, "meta-llama/llama-3.3-70b-instruct", [
            {"role": "system", "content": f"You are an OCR correction assistant for historical {language} newspaper texts. Fix only clear OCR errors like garbled characters, broken words, or obvious misreadings. Keep all names, places, dates, and historical terms exactly as they are. Return only the corrected text in {language}, nothing else. If the text is too garbled to correct confidently, return it exactly as is. Never invent or hallucinate replacement text."},
            {"role": "user", "content": f"Fix OCR errors: {chunk}"}
        ], 300)

        corrected.append(response.choices[0].message.content.strip())
        time.sleep(0.5)

    return ' '.join(corrected)

def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

def clean_dataset(input_path, output_path, language):
    """
    Cleans an entire dataset split with checkpoint saving.
    Resumes from where it left off if interrupted.
    """
    # Load source documents
    docs = load_jsonl(input_path)
    print(f"Loaded {len(docs)} documents from {input_path}")

    # Check for existing progress
    already_cleaned = []
    try:
        already_cleaned = load_jsonl(output_path)
        print(f"Resuming: {len(already_cleaned)}/{len(docs)} already cleaned")
    except:
        print("Starting from scratch")

    already_done_ids = {doc["document_id"] for doc in already_cleaned}
    docs_remaining = [doc for doc in docs if doc["document_id"] not in already_done_ids]
    print(f"Remaining: {len(docs_remaining)} documents")

    total_docs = len(docs)
    start_idx = len(already_cleaned)

    for doc_idx, doc in enumerate(docs_remaining):
        real_idx = start_idx + doc_idx + 1
        print(f"\n[{real_idx}/{total_docs}] Cleaning: {doc['document_id']}")
        doc["text"] = clean_text_llm(doc["text"], client, language=language)

        # Checkpoint save after every document
        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(doc) + "\n")

        time.sleep(1)

    print(f"\n✅ Done! Saved to {output_path}")

In [ ]:
# ============================================================
# 3. CONFIGURATION — edit paths and languages here
# ============================================================

DRIVE_PATH = "/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox"

splits = [
    # (input_file, output_file, language)
    (f"{DRIVE_PATH}/en-dev.jsonl",   f"{DRIVE_PATH}/en-dev-cleaned.jsonl",   "English"),
    (f"{DRIVE_PATH}/en-train.jsonl", f"{DRIVE_PATH}/en-train-cleaned.jsonl", "English"),
    (f"{DRIVE_PATH}/fr-dev.jsonl",   f"{DRIVE_PATH}/fr-dev-cleaned.jsonl",   "French"),
    (f"{DRIVE_PATH}/fr-train.jsonl", f"{DRIVE_PATH}/fr-train-cleaned.jsonl", "French"),
    (f"{DRIVE_PATH}/de-dev.jsonl",   f"{DRIVE_PATH}/de-dev-cleaned.jsonl",   "German"),
    (f"{DRIVE_PATH}/de-train.jsonl", f"{DRIVE_PATH}/de-train-cleaned.jsonl", "German"),
]

In [ ]:
# ============================================================
# 4. RUN CLEANING — uncomment the split you want to process
# ============================================================

# clean_dataset(*splits[0])  # EN dev
# clean_dataset(*splits[1])  # EN train
# clean_dataset(*splits[2])  # FR dev
# clean_dataset(*splits[3])  # FR train
# clean_dataset(*splits[4])  # DE dev
# clean_dataset(*splits[5])  # DE train